In [1]:
import pandas as pd
import warnings
import numpy as np
import copy
from datetime import date

warnings.filterwarnings("ignore")

project_name='LAGUARDIA'

# file_name='LAGUARDIA_KINGElvis_no_aa.xlsx'
file_name='LAGUARDIA_KINGElvis_no_aa.csv'
today_date = date.today()
today_date=''.join(str(today_date).split('-'))

elvis_df=pd.read_csv(file_name)

In [2]:
try:
    traditional_df=pd.read_csv(f'reviewtool_{today_date}_{project_name}_TraditionalTransferFlags.csv')
except:
    traditional_df=pd.DataFrame()
try:
    od_df=pd.read_csv(f'reviewtool_{today_date}_{project_name}_OD_Distance_Checks.csv')
except:
    od_df=pd.DataFrame()
try:
    transfer_df=pd.read_csv(f'reviewtool_{today_date}_{project_name}_Distance_Transfer_Flags.csv')
except:
    transfer_df=pd.DataFrame()

In [3]:

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns



if not traditional_df.empty:
    # merge Traditional Transfer Checks with KingELvis Data
    merged_df = pd.merge(elvis_df, traditional_df['id'], on='id', how='left', indicator=True)
    # Create a new column 'Traditional_Check' based on the merge indicator
    merged_df['Traditional_Check'] = (merged_df['_merge'] == 'both').astype(int)
    # Drop the indicator column and display the resulting DataFrame
    merged_df = merged_df.drop(columns=['_merge'])
else:
    merged_df=copy.deepcopy(elvis_df)
    merged_df['Traditional_Check']=0

In [4]:
if not od_df.empty:
    # merge OD Distance Transfer Checks with Merged Data of Traditional Checks
    merged_df = pd.merge(merged_df, od_df[['id','O_D_Distance_Flag_Description']], on='id', how='left', indicator=True)
    # Create a new column 'OD_Distance_Check' based on the merge indicator
    merged_df['OD_Distance_Check'] = (merged_df['_merge'] == 'both').astype(int)
    # Drop the indicator column and display the resulting DataFrame
    merged_df = merged_df.drop(columns=['_merge'])
else:
    merged_df['OD_Distance_Check']=0

if not transfer_df.empty:
    # merge TRansfer Distance Checks with Merged Data of Traditional and OD Distance Checks
    merged_df = pd.merge(merged_df, transfer_df[['id','Transfer_Distance_Flag_Description']], on='id', how='left', indicator=True)
    # Create a new column 'Transfer_Distance_Check' based on the merge indicator
    merged_df['Transfer_Distance_Check'] = (merged_df['_merge'] == 'both').astype(int)
    # Drop the indicator column and display the resulting DataFrame
    merged_df = merged_df.drop(columns=['_merge'])
else:
    merged_df['Transfer_Distance_Check']=0

In [5]:
merged_df

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,Final_Usage,FINAL_REVIEWER,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],POSSIBLE ERRORS,distance_flag,...,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,O_D_Distance_Flag_Description,OD_Distance_Check,Transfer_Distance_Flag_Description,Transfer_Distance_Check
0,2024-08-19,514,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,NaN,0,NaN,0
1,2024-08-19,522,HereAPI,,Use,,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,Origin to Board distance is greater than 1.85 ...,1,NaN,0
2,2024-08-19,524,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,NaN,0,NaN,0
3,2024-08-19,525,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,Approved and OK to reverse,NaN,Elvis_Review,Elvis_Review,Tested,0,NaN,0,NaN,0
4,2024-08-19,528,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,2024-08-19,1874,HereAPI,,Use,,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,Origin to Board distance is less than 0.25 mil...,1,NaN,0
952,2024-08-19,1875,HereAPI,,Use,,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,Origin to Board distance is less than 0.25 mil...,1,One/More Transfer Points Missing,1
953,2024-08-19,1876,HereAPI,,Use,,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,Origin to Board distance is less than 0.25 mil...,1,One/More Transfer Points Missing,1
954,2024-08-19,1877,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,0,NaN,0,NaN,0


In [6]:

flag_description_columns_check=['oddistanceflagdescription','transferdistanceflagdescription']
flag_description_columns=check_all_characters_present(merged_df,flag_description_columns_check)

if flag_description_columns:
    for _,row in merged_df.iterrows():
        if not pd.isna(row['O_D_Distance_Flag_Description']) and not pd.isna(row['Transfer_Distance_Flag_Description']):
            merged_df.loc[row.name,'Flag Description']=row['O_D_Distance_Flag_Description'] +' '+ row['Transfer_Distance_Flag_Description']
        elif not pd.isna(row['O_D_Distance_Flag_Description']):
            merged_df.loc[row.name,'Flag Description']=row['O_D_Distance_Flag_Description']
        elif not pd.isna(row['Transfer_Distance_Flag_Description']):
            merged_df.loc[row.name,'Flag Description']= row['Transfer_Distance_Flag_Description']
        else:
            merged_df.loc[row.name,'Flag Description']=' '
else:
    merged_df['Flag Description']=' '

merged_df.drop(columns=['Transfer_Distance_Flag_Description','O_D_Distance_Flag_Description'],inplace=True)

review_columns_check=['2xreviewcheck','2X_REVIEW_CHECK.1']

review_columns=check_all_characters_present(elvis_df,review_columns_check)

In [7]:
if review_columns:
    # create new column where all checks are 1
    merged_df.drop(columns=[*review_columns],inplace=True)
    merged_df['2X_REVIEW_CHECK']=np.where(merged_df[['Transfer_Distance_Check','OD_Distance_Check','Traditional_Check']].any(axis=1),1,0)
    # merged_df['2X_REVIEW_CHECK']=np.where(merged_df[['Transfer_Distance_Check','OD_Distance_Check','Traditional_Check','Recovery_Check']].any(axis=1),1,0)
else:
    merged_df['2X_REVIEW_CHECK']=np.where(merged_df[['Transfer_Distance_Check','OD_Distance_Check','Traditional_Check']].any(axis=1),1,0)
    # create new empty columns
    merged_df['2x_REVIEWED_BY']=None
    merged_df['2x_REVIEWED_FLAG']=None
    merged_df['ADMIN_APPROVED']=None

# merged_df[merged_df['2X_REVIEW_CHECK']==1].to_csv('Merged Checks.csv',index=False)

merged_df.drop_duplicates(subset=['id'],inplace=True)

In [9]:
database_df=pd.read_csv('elvislaguardia2024intercept_export_odbc_filled.csv')
database_df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_DATE,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,514,2024-07-24 13:56:09,87.0,en,2024-07-24 13:51:11,2024-07-24 13:56:09,50.202.171.186,https://laguardia-ny.etc-research.com/,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,522,2024-07-24 14:14:10,87.0,en,2024-07-24 14:06:11,2024-07-24 14:14:10,50.202.171.186,NaN,-14400,MDG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,524,2024-07-24 14:17:42,87.0,en,2024-07-24 14:16:08,2024-07-24 14:17:42,50.202.171.186,https://laguardia-ny.etc-research.com/index.php,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,525,2024-07-24 14:23:47,87.0,en,2024-07-24 14:17:44,2024-08-01 08:26:15,50.202.171.186,https://laguardia-ny.etc-research.com/index.ph...,-14400,999,...,\n\n,\n\n,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,n1
4,528,2024-07-24 14:30:50,87.0,en,2024-07-24 14:19:21,2024-07-24 14:30:50,12.1.48.108,https://laguardia-ny.etc-research.com/,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
merged_df.columns

Index(['Elvis_Date', 'elvis_id', '1st Cleaner', '2nd Cleaner', 'Final_Usage',
       'FINAL_REVIEWER', 'REASON FOR REMOVAL', 'REASON FOR REMOVAL [Other]',
       'POSSIBLE ERRORS', 'distance_flag', 'route_match_flag',
       'SUPERVISOR_COMMENT', 'id', 'Completed', 'INTERV_INIT',
       'ROUTE_SURVEYEDCode', 'ROUTE_SURVEYED', 'HAVE_5_MIN_FOR_SURVECode',
       'HAVE_5_MIN_FOR_SURVE', 'ORIGIN_PLACE_TYPE', 'ORIGIN_TRANSPORTCode',
       'ORIGIN_TRANSPORT', 'DESTIN_PLACE_TYPE', 'DESTIN_TRANSPORTCode',
       'DESTIN_TRANSPORT', 'ELVIS_STATUS', 'ELVIS_COMMENT', 'ROUTE_STATUS',
       'Stops_Status', 'Test_Status', 'Traditional_Check', 'OD_Distance_Check',
       'Transfer_Distance_Check', 'Flag Description', '2X_REVIEW_CHECK',
       '2x_REVIEWED_BY', '2x_REVIEWED_FLAG', 'ADMIN_APPROVED'],
      dtype='object')

In [11]:
od_ba_names_check=['originaddresslat', 'originaddresslong', 'stoponlat', 'stoponlong', 'stopofflat', 'stopofflong', 'destinaddresslat', 'destinaddresslong']
od_ba_names=check_all_characters_present(database_df,od_ba_names_check)
od_ba_names

['ORIGIN_ADDRESS_LAT',
 'ORIGIN_ADDRESS_LONG',
 'DESTIN_ADDRESS_LAT',
 'DESTIN_ADDRESS_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG']

In [14]:
new_df=pd.merge(merged_df,database_df[['id',*od_ba_names]],on='id',how='left')

In [15]:
new_df.head()

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,Final_Usage,FINAL_REVIEWER,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],POSSIBLE ERRORS,distance_flag,...,2x_REVIEWED_FLAG,ADMIN_APPROVED,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,STOP_ON_LAT,STOP_ON_LONG,STOP_OFF_LAT,STOP_OFF_LONG
0,2024-08-19,514,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,None,None,40.726545,-73.941539,40.772924,-73.872198,40.768790,-73.910815,40.772924,-73.872198
1,2024-08-19,522,HereAPI,,Use,,,,,Elvis_Review,...,None,None,40.773656,-73.959860,40.772924,-73.872198,40.801818,-73.967642,40.772924,-73.872198
2,2024-08-19,524,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,None,None,40.772924,-73.872198,40.772924,-73.872198,40.772924,-73.872198,40.772924,-73.872198
3,2024-08-19,525,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,None,None,40.772924,-73.872198,40.747929,-73.943983,40.772924,-73.872198,40.773656,-73.959860
4,2024-08-19,528,Test,,Remove,Test/No 5 MIN,,,,Elvis_Review,...,None,None,40.706672,-73.960228,40.772924,-73.872198,40.746543,-73.890686,40.772924,-73.872198


In [16]:
cols = list(new_df.columns)

# Step 3: Remove the new columns from their current position
for col in od_ba_names:
    cols.remove(col)

In [17]:
cols

['Elvis_Date',
 'elvis_id',
 '1st Cleaner',
 '2nd Cleaner',
 'Final_Usage',
 'FINAL_REVIEWER',
 'REASON FOR REMOVAL',
 'REASON FOR REMOVAL [Other]',
 'POSSIBLE ERRORS',
 'distance_flag',
 'route_match_flag',
 'SUPERVISOR_COMMENT',
 'id',
 'Completed',
 'INTERV_INIT',
 'ROUTE_SURVEYEDCode',
 'ROUTE_SURVEYED',
 'HAVE_5_MIN_FOR_SURVECode',
 'HAVE_5_MIN_FOR_SURVE',
 'ORIGIN_PLACE_TYPE',
 'ORIGIN_TRANSPORTCode',
 'ORIGIN_TRANSPORT',
 'DESTIN_PLACE_TYPE',
 'DESTIN_TRANSPORTCode',
 'DESTIN_TRANSPORT',
 'ELVIS_STATUS',
 'ELVIS_COMMENT',
 'ROUTE_STATUS',
 'Stops_Status',
 'Test_Status',
 'Traditional_Check',
 'OD_Distance_Check',
 'Transfer_Distance_Check',
 'Flag Description',
 '2X_REVIEW_CHECK',
 '2x_REVIEWED_BY',
 '2x_REVIEWED_FLAG',
 'ADMIN_APPROVED']

In [18]:
insert_position = 3

# Step 5: Insert the new columns into the desired position
for col in reversed(od_ba_names):  # reversed to maintain order when inserting
    cols.insert(insert_position, col)



In [19]:
cols

['Elvis_Date',
 'elvis_id',
 '1st Cleaner',
 'ORIGIN_ADDRESS_LAT',
 'ORIGIN_ADDRESS_LONG',
 'DESTIN_ADDRESS_LAT',
 'DESTIN_ADDRESS_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 '2nd Cleaner',
 'Final_Usage',
 'FINAL_REVIEWER',
 'REASON FOR REMOVAL',
 'REASON FOR REMOVAL [Other]',
 'POSSIBLE ERRORS',
 'distance_flag',
 'route_match_flag',
 'SUPERVISOR_COMMENT',
 'id',
 'Completed',
 'INTERV_INIT',
 'ROUTE_SURVEYEDCode',
 'ROUTE_SURVEYED',
 'HAVE_5_MIN_FOR_SURVECode',
 'HAVE_5_MIN_FOR_SURVE',
 'ORIGIN_PLACE_TYPE',
 'ORIGIN_TRANSPORTCode',
 'ORIGIN_TRANSPORT',
 'DESTIN_PLACE_TYPE',
 'DESTIN_TRANSPORTCode',
 'DESTIN_TRANSPORT',
 'ELVIS_STATUS',
 'ELVIS_COMMENT',
 'ROUTE_STATUS',
 'Stops_Status',
 'Test_Status',
 'Traditional_Check',
 'OD_Distance_Check',
 'Transfer_Distance_Check',
 'Flag Description',
 '2X_REVIEW_CHECK',
 '2x_REVIEWED_BY',
 '2x_REVIEWED_FLAG',
 'ADMIN_APPROVED']

In [20]:
new_df = new_df[cols]

In [21]:
new_df

,Elvis_Date,elvis_id,1st Cleaner,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,STOP_ON_LAT,STOP_ON_LONG,STOP_OFF_LAT,...,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,Flag Description,2X_REVIEW_CHECK,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED
0,2024-08-19,514,Test,40.726545,-73.941539,40.772924,-73.872198,40.768790,-73.910815,40.772924,...,Elvis_Review,Tested,0,0,0,,0,None,None,None
1,2024-08-19,522,HereAPI,40.773656,-73.959860,40.772924,-73.872198,40.801818,-73.967642,40.772924,...,Elvis_Review,Tested,0,1,0,Origin to Board distance is greater than 1.85 ...,1,None,None,None
2,2024-08-19,524,Test,40.772924,-73.872198,40.772924,-73.872198,40.772924,-73.872198,40.772924,...,Elvis_Review,Tested,0,0,0,,0,None,None,None
3,2024-08-19,525,Test,40.772924,-73.872198,40.747929,-73.943983,40.772924,-73.872198,40.773656,...,Elvis_Review,Tested,0,0,0,,0,None,None,None
4,2024-08-19,528,Test,40.706672,-73.960228,40.772924,-73.872198,40.746543,-73.890686,40.772924,...,Elvis_Review,Tested,0,0,0,,0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,2024-08-19,1874,HereAPI,40.772924,-73.872198,40.806112,-73.944892,40.772924,-73.872198,40.808026,...,Elvis_Review,Tested,0,1,0,Origin to Board distance is less than 0.25 mil...,1,None,None,None
952,2024-08-19,1875,HereAPI,40.772924,-73.872198,40.744498,-73.914633,40.772924,-73.872198,40.745704,...,Elvis_Review,Tested,0,1,1,Origin to Board distance is less than 0.25 mil...,1,None,None,None
953,2024-08-19,1876,HereAPI,40.772924,-73.872198,40.756697,-73.961160,40.772924,-73.872198,40.746819,...,Elvis_Review,Tested,0,1,1,Origin to Board distance is less than 0.25 mil...,1,None,None,None
954,2024-08-19,1877,Test,40.772924,-73.872198,40.747929,-73.943983,40.772924,-73.872198,40.773730,...,Elvis_Review,Tested,0,0,0,,0,None,None,None
